# DX 704 Week 11 Project | Prompting an LLM

In this project, you will develop and test prompts asking a language model to classify text from a home services query and match it to an appropriate category of home services.

The full project description and a template notebook are available on GitHub: [Project 11 Materials](https://github.com/bu-cds-dx704/dx704-project-11).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Imports


In [21]:
import pandas as pd
import numpy as np
import re
import csv

%pip install -q -U google-genai
# pip install -q -U google-genai
import google.genai as genai
# from google.colab import userdata


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
my_key = ''
client = genai.Client(api_key = my_key)  
# client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))  #enable if using CoLab

In [23]:
# Rate limit documentation https://ai.google.dev/gemini-api/docs/rate-limits
model_name = 'gemini-2.0-flash'

## Functions

In [24]:
def get_response(contents):
  response = client.models.generate_content(model     = model_name,
                                            contents  = contents
                                            )
  return response.text

In [25]:
def get_boolean_response(contents):
  response = get_response(contents)
  print("RESPONSE", response)

  output = response.lower().startswith("yes")
  print("OUTPUT", output)

  return output

In [26]:

def classify_query(query, prompt:str, display_prompt = False) -> str:
  """
  Formats the short prompt, gets the model response, then cleans it by lower casing and only returning the first line of the response
  in case there are multiple lines.
  """

  # pass the query using string formatting into the prompt
  full_prompt = prompt.format(query = query)
  if display_prompt:
        print(f'Full prompt: {full_prompt}')


  # Get raw response from the model
  raw_prediction = get_response(full_prompt)
  print (f'raw_prediction: {raw_prediction}')

  # Clean prediction to standardize output
  cleaned_prediction = raw_prediction.strip().lower().split('\n')[0].strip()
  print (f'cleaned_prediction: {cleaned_prediction}\n----------------------------------')

  # check: prediction needs to match one of the three categories
  categories = ['electrical', 'plumbing', 'roofing']
  if cleaned_prediction not in categories:
      # if model adds extra words,  extract the first valid category
      for cat in categories:
          if cat in cleaned_prediction:
              return cat
      # Default to 'unknown' if no valid category is found after cleaning
      return 'unknown'

  return cleaned_prediction



## Part 1 : Design a Short Prompt

The provided file "`queries.txt`" contains sample text from requests by homeowners by email or phone.
- These queries need to be classified as requesting **electrical**, **plumbing**, or **roofing**  services.
- The provided file has columns `query_id`, `query`, and `target_category`.
- Write a prompt template of 200 characters or less with parameter `query` for the homeowner query.
- Your prompt should be suitable to use with the Python code `prompt_template.format(query=query)`.
- Test your prompt with the model `gemini-2.0-flash` and suitable parsing code.


- Save your prompt template in a file "`short-prompt.txt`".
- Save the results of your prompt testing in "`short-output.tsv`" with columns `query_id` and `predicted_category`.
- Hint: your prompt may be re-tested with the Gemini API, so do not rely solely on lucky language model responses.

In [40]:
# YOUR CHANGES HERE

PROMPT_TEMPLATE = 'short-prompt.txt'

my_short_prompt = 'Classify this homeowner request into electrical, plumbing, or roofing:\n     "{query}".\nReturn only one category.'
with open(PROMPT_TEMPLATE, 'w') as f:
    f.write(my_short_prompt)

queries = []
with open('queries.txt', 'r') as f:
    reader = csv.DictReader(f, delimiter = '\t')
    for row in reader:
        queries.append(
            {
            'query_id': row['query_id'],
            'query':    row['query']
            }
        )

queries_df = pd.DataFrame(queries)
queries_df

,query_id,query
0,1,Hi. Melissa came by and wrecked my roof. Can y...
1,2,Hi there. This is Jack. I’m looking for someon...
2,3,Can you install an automated spotlight by my d...
3,4,Pest control just cleared out a raccoon that t...
4,5,Need toilet unclogged ASAP
5,6,My lights keep flickering.
6,7,Do you install metal roofs?
7,8,Can you fix a garbage disposal that keeps back...
8,9,Need to install 200 amp circuit to support ele...
9,10,What’s the cost of a single ply roof replacement?


In [28]:
# YOUR CHANGES HERE
results = []
print(len(queries_df))

50


In [29]:

start_index = 0
end_index   = len(queries_df)

# Itertuples.. how the hell did i not know about this ??? i can iterate over each row in the dataframe. 
for row in queries_df.iloc[start_index:end_index].itertuples():
    # pass the query in the current row to classify_query
    print(f'query_id: {row.query_id}')
    predicted_category = classify_query(row.query, prompt = my_short_prompt, display_prompt = True)

    results.append({
        'query_id':           row.query_id,
        'predicted_category': predicted_category
    })


query_id: 1
Full prompt: Classify this homeowner request into electrical, plumbing, or roofing: "Hi. Melissa came by and wrecked my roof. Can you take a look?". Return only one category.
raw_prediction: Roofing

cleaned_prediction: roofing
----------------------------------
query_id: 2
Full prompt: Classify this homeowner request into electrical, plumbing, or roofing: "Hi there. This is Jack. I’m looking for someone to install a new sink faucet. Call me back, thanks.". Return only one category.
raw_prediction: Plumbing

cleaned_prediction: plumbing
----------------------------------
query_id: 3
Full prompt: Classify this homeowner request into electrical, plumbing, or roofing: "Can you install an automated spotlight by my driveway?". Return only one category.
raw_prediction: Electrical

cleaned_prediction: electrical
----------------------------------
query_id: 4
Full prompt: Classify this homeowner request into electrical, plumbing, or roofing: "Pest control just cleared out a raccoon

In [30]:
output_df = pd.DataFrame(results)
output_df

,query_id,predicted_category
0,1,roofing
1,2,plumbing
2,3,electrical
3,4,roofing
4,5,plumbing
5,6,electrical
6,7,roofing
7,8,plumbing
8,9,electrical
9,10,roofing


In [31]:
output_file_name = 'short-output.tsv'
output_df.to_csv(output_file_name, sep = '\t', index = False)

Submit "`short-prompt.txt`" and "`short-output.tsv`" in Gradescope.

## Part 2: Find Short Prompt Mistakes

Construct 5 queries of 100 characters or less that trick your short prompt so that the wrong category is chosen.


Save your 5 queries in a file "mistakes.tsv" with columns `query`, `target_category` and `predicted_category`.

In [47]:
# YOUR CHANGES HERE

bad_query_1 = {'id': 1,
               'query': 'What up foo? Need somebody to raise the you-know-what at my next house party.',
               'target_category': 'roofing'
               }
bad_query_2 = {'id': 2,
               'query': 'Do you install solar panels ? ',
               'target_category': 'electrical'
              }

bad_query_3 = {'id': 3,
               'query':'I am going to sell my house and need my kitchen appliances inspected.',
               'target_category': 'plumbing'
              }

bad_query_4 = {'id': 4,
               'query': 'I want to install a satellite dish. ',
               'target_category': 'roofing'
               }

bad_query_5 = {
    'id': 5,
    'query': 'Hi, I need pest control in my house for areas not plumbing.',
    'target_category': 'electrical'
}

bad_queries = [bad_query_1, bad_query_2, bad_query_3, bad_query_4, bad_query_5]


In [48]:
# YOUR CHANGES HERE

for query in bad_queries:
    print(f"Target category: {query['target_category']}\n")
    predicted_category = classify_query(query['query'], 
                                        prompt = my_short_prompt, 
                                        display_prompt = True
                                        )

    query['predicted_category'] = predicted_category


Target category: roofing

Full prompt: Classify this homeowner request into electrical, plumbing, or roofing:
     "What up foo? Need somebody to raise the you-know-what at my next house party.".
Return only one category.
raw_prediction: Electrical

cleaned_prediction: electrical
----------------------------------
Target category: electrical

Full prompt: Classify this homeowner request into electrical, plumbing, or roofing:
     "Do you install solar panels ? ".
Return only one category.
raw_prediction: Roofing

cleaned_prediction: roofing
----------------------------------
Target category: plumbing

Full prompt: Classify this homeowner request into electrical, plumbing, or roofing:
     "I am going to sell my house and need my kitchen appliances inspected.".
Return only one category.
raw_prediction: Electrical

cleaned_prediction: electrical
----------------------------------
Target category: roofing

Full prompt: Classify this homeowner request into electrical, plumbing, or roofing:

In [49]:
prompt_mistakes_df = pd.DataFrame(bad_queries)
prompt_mistakes_df = prompt_mistakes_df[['query', 'target_category', 'predicted_category']]
prompt_mistakes_df

,query,target_category,predicted_category
0,What up foo? Need somebody to raise the you-kn...,roofing,electrical
1,Do you install solar panels ?,electrical,roofing
2,I am going to sell my house and need my kitche...,plumbing,electrical
3,I want to install a satellite dish.,roofing,electrical
4,"Hi, I need pest control in my house for areas ...",electrical,roofing


In [50]:
file_name = 'mistakes.tsv'
prompt_mistakes_df.to_csv(file_name, sep = '\t', index = False)

Submit "mistakes.tsv" in Gradescope.

## Part 3: Design a Long Prompt

Repeat part 1 with a length limit of 5000 characters.

In [52]:
PROMPT_LONG_TEMPLATE = 'long-prompt.txt'


my_longass_prompt = """You are an expert home services dispatcher. Your task: classify a homeowner's service request into one and only one of three categories: 'electrical', 'plumbing', or 'roofing'.

GUIDELINES:
1.  **Output Format:** Return ONLY the single, lowercase category name. Do not include any other text, punctuation, or explanation.
2.  **Categories and Definitions:**
    * **electrical:** Any request that involves lighting, ceiling fans, automated devices, or anything that draws power.
    * **plumbing:** Any request that involves water flow, drainage, sewage, fixtures, water heaters, boilers, pipes, leaks originating from an internal water source or basement flooding.
    * **roofing:** Any request that involves the exterior structure on top of the home.
CLASSIFY THE FOLLOWING HOMEOWNER REQUEST:\n"{query}"\n
Response:"""

with open(PROMPT_LONG_TEMPLATE, 'w') as f:
    f.write(my_longass_prompt)


results_3 = []

In [53]:
start_index = 0
end_index   = len(queries_df)

for row in queries_df.iloc[start_index:end_index].itertuples():
    # pass the query in the. current row

    print(f'query_id: {row.query_id}')
    predicted_category = classify_query(row.query, prompt = my_longass_prompt, display_prompt= False)

    results_3.append({
        'query_id':           row.query_id,
        'predicted_category': predicted_category
    })


query_id: 1
raw_prediction: roofing

cleaned_prediction: roofing
----------------------------------
query_id: 2
raw_prediction: plumbing

cleaned_prediction: plumbing
----------------------------------
query_id: 3
raw_prediction: electrical

cleaned_prediction: electrical
----------------------------------
query_id: 4
raw_prediction: roofing

cleaned_prediction: roofing
----------------------------------
query_id: 5
raw_prediction: plumbing

cleaned_prediction: plumbing
----------------------------------
query_id: 6
raw_prediction: electrical

cleaned_prediction: electrical
----------------------------------
query_id: 7
raw_prediction: roofing

cleaned_prediction: roofing
----------------------------------
query_id: 8
raw_prediction: plumbing

cleaned_prediction: plumbing
----------------------------------
query_id: 9
raw_prediction: electrical

cleaned_prediction: electrical
----------------------------------
query_id: 10
raw_prediction: roofing

cleaned_prediction: roofing
----------

In [54]:
long_output_df = pd.DataFrame(results_3)
long_output_df

,query_id,predicted_category
0,1,roofing
1,2,plumbing
2,3,electrical
3,4,roofing
4,5,plumbing
5,6,electrical
6,7,roofing
7,8,plumbing
8,9,electrical
9,10,roofing


Save your longer prompt template in a file "`long-prompt.txt`".
Save the results of your prompt testing in "`long-output.tsv`".
Both files should use the same columns as part 1.

In [55]:
# YOUR CHANGES HERE

file_name = 'long-output.tsv'
long_output_df.to_csv(file_name, sep = '\t', index = False)

Submit "long-prompt.txt" and "long-output.tsv" in Gradescope.

## Part 4: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 5: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.